01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
.appName("IcebergWithSpark") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
.config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
.config("spark.sql.default.catalog", "hadoop_catalog") \
.getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,951 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,945 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,011 kB]
Get:14 http://security.ubunt

In [27]:
from pyspark.sql.functions import to_date, col

# Versões e Rollbacks de Tabelas

In [38]:
# Criação da Tabela
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_versioned")

spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas_versioned (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")

DataFrame[]

In [39]:
# Dados
data_initial = [
    (1, "Produto A", 10, 15.5, "2023-11-01"),
    (2, "Produto B", 5, 22.0, "2023-11-02"),
    (3, "Produto C", 8, 30.0, "2023-11-03")
]
columns = ["id", "produto", "quantidade", "preco", "data_venda"]

df_initial = spark.createDataFrame(data_initial, columns)
df_initial = df_initial.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

df_initial.writeTo("hadoop_catalog.default.vendas_versioned").append()

In [40]:
# Visualização
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_versioned").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



In [41]:
# Criamos um novo snapshot inserindo mais dados
data_additional = [
    (4, "Produto D", 12, 25.0, "2023-11-04"),
    (5, "Produto E", 7, 18.5, "2023-11-05")
]
df_additional = spark.createDataFrame(data_additional, columns)
df_additional = df_additional.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))
df_additional.writeTo("hadoop_catalog.default.vendas_versioned").append()

In [42]:
# Atualizamos preço
spark.sql("""
    UPDATE hadoop_catalog.default.vendas_versioned
    SET preco = 16.0
    WHERE id = 1
""")

DataFrame[]

In [43]:
# Excluimos registro 2
spark.sql("""
    DELETE FROM hadoop_catalog.default.vendas_versioned
    WHERE id = 2
""")

DataFrame[]

In [44]:
# visualização
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_versioned order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [45]:
# Visualizamos os snapshots
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas_versioned.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1434122609968601888|2026-03-30 20:42:17.328|append   |
|8356733291729498889|2026-03-30 20:42:31.789|append   |
|5892506511127571584|2026-03-30 20:42:41.592|overwrite|
|8961227683902089124|2026-03-30 20:42:47.452|overwrite|
+-------------------+-----------------------+---------+



In [46]:
# Obtemos o ID do snapshot para fazer rollback, indíce 1
snapshot_ids = snapshots_df.select("snapshot_id").orderBy("committed_at").collect()
rollback_snapshot_id = snapshot_ids[1].snapshot_id
print(f"Snapshot ID: {rollback_snapshot_id}")

Snapshot ID: 8356733291729498889


In [47]:
print("Antes do Rollback:")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_versioned order by id").show()

Antes do Rollback:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [48]:
# Rollback usado rollback_to_snapshot procedure
spark.sql(f"""
    CALL hadoop_catalog.system.rollback_to_snapshot(
        table => 'hadoop_catalog.default.vendas_versioned',
        snapshot_id => {rollback_snapshot_id}
    )
""")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [49]:
print("Dados depois do Rollback:")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_versioned order by id").show()

Dados depois do Rollback:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+

